# Tier 1 — Modèles prêts à l'emploi : Détection visage + Âge + Sexe

**Projet** : Analyse faciale pour recommandation parapharmaceutique
**Phase** : Tier 1 (aucun entraînement — modèles pré-entraînés)
**Environnement** : Kaggle Notebook

Ce notebook met en place la première brique du pipeline :
1. Détection + alignement du visage
2. Estimation de l'**âge**
3. Classification du **sexe**

avec deux bibliothèques matures (**DeepFace** et **InsightFace**), puis une **fonction unifiée**
renvoyant une sortie structurée (JSON) directement réutilisable plus tard dans le backend FastAPI.

> ⚙️ **Avant de lancer** : dans le panneau de droite Kaggle →
> - **Settings → Accelerator : GPU** (T4 recommandé)
> - **Settings → Internet : ON** (obligatoire : les poids des modèles se téléchargent au 1er appel)


## 1. Installation des dépendances

Sur Kaggle, l'image de base contient déjà TensorFlow/PyTorch. On ajoute :
- `deepface` (+ `tf-keras` requis par les versions récentes)
- `insightface` + `onnxruntime-gpu` (inférence ONNX)


In [ ]:
!pip install -q deepface tf-keras insightface onnxruntime-gpu

# Si le GPU n'est pas activé, remplace par :  !pip install -q onnxruntime
print("Installation terminée.")

## 2. Imports & configuration

In [ ]:
import os
import glob
import json
import warnings
warnings.filterwarnings("ignore")

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Vérification GPU
import onnxruntime as ort
print("Providers ONNX disponibles :", ort.get_available_providers())

USE_GPU = "CUDAExecutionProvider" in ort.get_available_providers()
print("GPU utilisé :", USE_GPU)

## 3. Préparation des images

Sur Kaggle, les jeux de données montés se trouvent sous `/kaggle/input/...`
et les sorties dans `/kaggle/working/`.

👉 Ajoute tes images de test via **+ Add Input** (un dataset d'images de visages),
le notebook les détecte automatiquement.


In [ ]:
INPUT_DIR  = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working"

# Recherche récursive des images dans /kaggle/input
exts = ("*.jpg", "*.jpeg", "*.png", "*.webp")
images = []
for ext in exts:
    images += glob.glob(os.path.join(INPUT_DIR, "**", ext), recursive=True)

images = sorted(images)
print(f"{len(images)} image(s) trouvée(s).")
for p in images[:5]:
    print(" -", p)

if not images:
    print("\n⚠️  Aucune image. Ajoute un dataset via '+ Add Input' puis relance cette cellule.")

In [ ]:
def load_image_rgb(path):
    """Charge une image en RGB (OpenCV lit en BGR par défaut)."""
    img = cv2.imread(path)
    if img is None:
        raise ValueError(f"Image illisible : {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Image d'exemple pour les tests ci-dessous
sample_path = images[0] if images else None
print("Image d'exemple :", sample_path)

## 4. Détection + alignement du visage (InsightFace)

InsightFace fournit un détecteur **SCRFD** rapide et robuste, l'alignement,
et — dans le même modèle `buffalo_l` — les têtes **âge** et **genre**.
On initialise l'analyseur une seule fois (les poids se téléchargent au 1er appel).


In [ ]:
from insightface.app import FaceAnalysis

providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if USE_GPU else ["CPUExecutionProvider"]

face_app = FaceAnalysis(name="buffalo_l", providers=providers)
face_app.prepare(ctx_id=0 if USE_GPU else -1, det_size=(640, 640))
print("InsightFace prêt.")

In [ ]:
def detect_faces_insightface(path):
    """Retourne la liste des visages détectés (boîtes, âge, genre, score)."""
    img_rgb = load_image_rgb(path)
    faces = face_app.get(cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
    return img_rgb, faces

if sample_path:
    img_rgb, faces = detect_faces_insightface(sample_path)
    print(f"{len(faces)} visage(s) détecté(s).")
    for i, f in enumerate(faces):
        x1, y1, x2, y2 = f.bbox.astype(int)
        print(f"  Visage {i}: bbox=({x1},{y1},{x2},{y2}) "
              f"score={f.det_score:.2f}")

## 5. Âge + Sexe — DeepFace

Première approche, une ligne de code. DeepFace gère détection, alignement et inférence.
On force `enforce_detection=False` pour éviter un crash si aucun visage net n'est trouvé.


In [ ]:
from deepface import DeepFace

def analyze_deepface(path, detector_backend="retinaface"):
    """Renvoie une liste de dicts {age, sexe} par visage détecté."""
    results = DeepFace.analyze(
        img_path=path,
        actions=["age", "gender"],
        detector_backend=detector_backend,
        enforce_detection=False,
    )
    out = []
    for r in results:
        sexe = "Homme" if r["dominant_gender"] == "Man" else "Femme"
        out.append({"age": int(r["age"]), "sexe": sexe})
    return out

if sample_path:
    print("DeepFace :", analyze_deepface(sample_path))

## 6. Âge + Sexe — InsightFace (alternative robuste)

Mêmes informations, mais via le modèle déjà chargé en section 4 — plus rapide,
souvent plus stable en conditions réelles, et fournit aussi les boîtes/landmarks
qui serviront à l'extraction des ROI au Tier 3.


In [ ]:
def analyze_insightface(path):
    """Renvoie une liste de dicts {age, sexe, bbox, score} par visage."""
    _, faces = detect_faces_insightface(path)
    out = []
    for f in faces:
        # f.gender : 1 = homme, 0 = femme ; f.age : entier
        sexe = "Homme" if int(f.gender) == 1 else "Femme"
        out.append({
            "age": int(f.age),
            "sexe": sexe,
            "bbox": f.bbox.astype(int).tolist(),
            "score": float(f.det_score),
        })
    return out

if sample_path:
    print("InsightFace :", analyze_insightface(sample_path))

## 7. Fonction unifiée (sortie structurée)

On encapsule le tout dans **une fonction réutilisable** qui renvoie un dictionnaire
au format aligné avec le futur profil client (champ `demographics`).
C'est cette fonction que tu porteras quasi telle quelle dans l'endpoint FastAPI `/v1/analyze`.

`engine` permet de choisir le moteur ; InsightFace par défaut.


In [ ]:
def analyze_demographics(path, engine="insightface"):
    """Analyse Tier 1 d'une image -> dict structuré (1er visage = principal)."""
    if engine == "deepface":
        faces = analyze_deepface(path)
        faces = [{"age": f["age"], "sexe": f["sexe"]} for f in faces]
    else:
        faces = analyze_insightface(path)

    if not faces:
        return {"ok": False, "raison": "aucun_visage_detecte", "image": os.path.basename(path)}

    principal = faces[0]  # visage principal = premier détecté
    return {
        "ok": True,
        "image": os.path.basename(path),
        "moteur": engine,
        "nb_visages": len(faces),
        "demographics": {
            "age_estime": principal["age"],
            "sexe": principal["sexe"],
        },
    }

if sample_path:
    profil = analyze_demographics(sample_path, engine="insightface")
    print(json.dumps(profil, indent=2, ensure_ascii=False))

## 8. Visualisation

On affiche l'image avec la boîte du visage et le résultat (âge / sexe) en surimpression.


In [ ]:
def show_result(path, engine="insightface"):
    img_rgb = load_image_rgb(path)
    faces = analyze_insightface(path)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(img_rgb)
    ax.axis("off")

    for f in faces:
        x1, y1, x2, y2 = f["bbox"]
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                    fill=False, edgecolor="#1ABC9C", linewidth=2))
        ax.text(x1, max(y1 - 8, 0), f"{f['sexe']}, {f['age']} ans",
                color="white", fontsize=11,
                bbox=dict(facecolor="#1ABC9C", edgecolor="none", pad=2))
    plt.tight_layout()
    plt.show()

if sample_path:
    show_result(sample_path)

## 9. Traitement par lot + export CSV

On applique l'analyse à toutes les images trouvées et on exporte un tableau
récapitulatif dans `/kaggle/working/resultats_tier1.csv`.


In [ ]:
rows = []
for p in images:
    try:
        r = analyze_demographics(p, engine="insightface")
        if r["ok"]:
            rows.append({
                "image": r["image"],
                "nb_visages": r["nb_visages"],
                "age_estime": r["demographics"]["age_estime"],
                "sexe": r["demographics"]["sexe"],
            })
        else:
            rows.append({"image": r["image"], "nb_visages": 0,
                         "age_estime": None, "sexe": None})
    except Exception as e:
        rows.append({"image": os.path.basename(p), "nb_visages": None,
                     "age_estime": None, "sexe": f"ERREUR: {e}"})

df = pd.DataFrame(rows)
out_csv = os.path.join(OUTPUT_DIR, "resultats_tier1.csv")
df.to_csv(out_csv, index=False)
print("Export :", out_csv)
df.head(20)

## 10. Validation & prochaines étapes

**Pour valider le Tier 1 avant de continuer :**
- Compare l'âge prédit à l'âge réel sur un petit échantillon étiqueté → calcule la **MAE**.
- Vérifie la robustesse sur **différents phototypes** et **conditions d'éclairage** (biais connu).
- Si la MAE est acceptable → on garde ces modèles tels quels (aucun entraînement).
- Sinon → fine-tuning léger sur UTKFace/IMDB-WIKI (passage en Tier 2 pour l'âge).

**Ensuite (Tier 2)** : attributs (barbe / moustache / lunettes) via CelebA, et type de cheveux.

**Réutilisation** : la fonction `analyze_demographics()` est l'ébauche directe de
l'endpoint FastAPI `/v1/analyze`. Côté serveur, on chargera `FaceAnalysis` une seule fois
au démarrage (et non par requête).
